# Classical Haufe mapping and permutation comparison

输出目录：`tmpt/CPM/CPM_classic_Haufe`。
本 notebook 使用已完成的逐折模型及置换缓存，默认不重新拟合。

## 1. 分析定义
在每个 outer LOOCV 训练折的入选、残差化并标准化特征空间中：
\[
a_j=\operatorname{Cov}(X_j,\hat y)/\operatorname{Var}(\hat y)
=(\Sigma_Xw)_j/\operatorname{Var}(\hat y).
\]
这是单输出线性模型的经典 Haufe pattern。未入选边记零；模块内求和后除以全部可能边数，再跨折平均。
Correlation 版本只用于匹配复现和比较，不能将其 P 值用于 slope。

组内检验：10,000 次站点内 IAT 残差置换，逐折重新选边、ridge 拟合和 Haufe 估计；固定原逐折 ridge alpha。
组间差值检验是探索性的联合“无 FC–IAT 关联”零假设检验，**不直接检验两组总体 Haufe 模式相等**。
右尾方向 high − low > 0 为事后探索方向；同时保留双侧结果。

In [1]:
from pathlib import Path
import sys, json, subprocess
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'Scripts/biox').is_dir())
SOURCE_DIR = PROJECT_ROOT / 'tmpt/CPM/CPM_nestedCV_LOOCV'
RESULT_DIR = PROJECT_ROOT / 'tmpt/CPM/CPM_classic_Haufe'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
GROUPS = ['highIAT', 'lowIAT']
N_PERMUTATIONS = 10000  # 可改为 5000，读取相同序列的前 5000 次
RUN_MISSING_PERMUTATIONS = False
BATCH_SIZE = 1000
N_JOBS = 4
assert N_PERMUTATIONS in (5000, 10000)


## 2. 可选：补算缺失置换
复用 `Scripts/compare_classic_haufe_permutation.py` 的原始数据重建、逐折处理和断点续算。
该脚本调用 `biox` 的 FC 函数，采用数学等价的 primal/dual ridge 解。
只有显式将上面的开关设为 True 才运行；现有完整 batch 和 fold 会跳过。
原始数据重建路径由该脚本的 `RAW` 指定，目前为 `~/Desktop/YiZhang`。

In [2]:
if RUN_MISSING_PERMUTATIONS:
    subprocess.run([
        sys.executable, '-u', str(PROJECT_ROOT / 'Scripts/compare_classic_haufe_permutation.py'),
        '--n-perm', str(N_PERMUTATIONS), '--batch', str(BATCH_SIZE), '--jobs', str(N_JOBS)
    ], cwd=PROJECT_ROOT, check=True)
else:
    print('仅读取已有缓存，不启动模型拟合。')

仅读取已有缓存，不启动模型拟合。


## 3. 读取并校验观察值和置换分布
`values[:, 0, :]` 为 correlation，`values[:, 1, :]` 为经典 slope。
仅接纳覆盖所有训练折的完整 batch，不把部分训练折当作完整置换。
检查编号无缺失、无重复，模块顺序一致，并从边级源表重建观察值。

In [3]:
observed_by_group, null_by_group = {}, {}
block_labels = None
for group in GROUPS:
    batch_values, batch_indices = [], []
    reference_observed = None
    for path in sorted(RESULT_DIR.glob(f'{group}_null_*.npz')):
        with np.load(path, allow_pickle=False) as z:
            keep = z['indices'] < N_PERMUTATIONS
            if not keep.any():
                continue
            labels = z['labels'].astype(str)
            if block_labels is None:
                block_labels = labels
            np.testing.assert_array_equal(labels, block_labels)
            if reference_observed is None:
                reference_observed = z['observed'].copy()
            np.testing.assert_allclose(z['observed'], reference_observed)
            batch_indices.extend(z['indices'][keep].tolist())
            batch_values.append(z['values'][keep].copy())
    if not batch_values:
        raise RuntimeError(f'{group}: 没有完整置换 batch；需先补算。')
    order = np.argsort(batch_indices)
    np.testing.assert_array_equal(np.asarray(batch_indices)[order], np.arange(N_PERMUTATIONS))
    null = np.concatenate(batch_values)[order]
    assert null.shape == (N_PERMUTATIONS, 2, 36) and np.isfinite(null).all()
    edges = pd.read_csv(SOURCE_DIR / f'CPM_nestedCV_{group}_haufe_all_FCs.csv')
    edges['block_label'] = edges.network_pair.str.replace(' - ', '-', regex=False)
    columns = ['haufe_corr_selected_mean_all_folds', 'haufe_slope_selected_mean_all_folds']
    observed = edges.groupby('block_label')[columns].mean().loc[block_labels].to_numpy().T
    np.testing.assert_allclose(observed, reference_observed, rtol=1e-9, atol=1e-12)
    observed_by_group[group], null_by_group[group] = reference_observed, null
    print(f'{group}: {len(null):,} complete permutations; 36 blocks; observed values verified.')

highIAT: 10,000 complete permutations; 36 blocks; observed values verified.
lowIAT: 10,000 complete permutations; 36 blocks; observed values verified.


## 4. 组内模块检验：双侧 P 和组内 36 项 BH-FDR

In [4]:
def bh_fdr(p_values):
    """Benjamini–Hochberg 校正，一次传入一个明确的检验族。"""
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    q = np.empty_like(p)
    q[order] = np.minimum(1, np.minimum.accumulate(
        (p[order] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1])
    return q

rows = []
for group in GROUPS:
    for m, mapping in enumerate(['correlation', 'classic_slope']):
        observed = observed_by_group[group][m]
        null = null_by_group[group][:, m, :]
        p = (1 + (np.abs(null) >= np.abs(observed)).sum(axis=0)) / (N_PERMUTATIONS + 1)
        q = bh_fdr(p)
        for j, label in enumerate(block_labels):
            rows.append(dict(group=group, mapping=mapping, network_block=label,
                value=observed[j], p_two_sided=p[j], q_fdr=q[j],
                n_permutations=N_PERMUTATIONS))
within_group = pd.DataFrame(rows)
within_group.to_csv(RESULT_DIR / f'classic_notebook_within_group_{N_PERMUTATIONS}perm.csv', index=False)
display(within_group.loc[within_group.p_two_sided < .05].round(6))
print('FDR-significant blocks:', int((within_group.q_fdr < .05).sum()))

,group,mapping,network_block,value,p_two_sided,q_fdr,n_permutations
18,highIAT,correlation,DAN-FPN,0.013506,0.020798,0.575942,10000
24,highIAT,correlation,VAN-DMN,0.009389,0.047695,0.575942,10000
54,highIAT,classic_slope,DAN-FPN,0.002838,0.014999,0.539946,10000
55,highIAT,classic_slope,DAN-DMN,0.001906,0.047895,0.574743,10000
60,highIAT,classic_slope,VAN-DMN,0.001971,0.045895,0.574743,10000
99,lowIAT,correlation,Lim-FPN,-0.009419,0.039496,1.000000,10000


FDR-significant blocks: 0


## 5. 探索性模块差值：双侧和右尾
两组同编号置换值相减，非被试配对。
P = (1 + 尾部计数)/(B + 1)，不是简单把双侧 P 除以 2。
每个映射、每种检验方向分别对全部 36 个模块做 BH-FDR。
右尾的 high > low 指带符号值更大，不是绝对强度更大。

In [5]:
rows = []
for m, mapping in enumerate(['correlation', 'classic_slope']):
    delta = observed_by_group['highIAT'][m] - observed_by_group['lowIAT'][m]
    null_delta = null_by_group['highIAT'][:, m] - null_by_group['lowIAT'][:, m]
    p_two = (1 + (np.abs(null_delta) >= np.abs(delta)).sum(axis=0)) / (N_PERMUTATIONS + 1)
    p_right = (1 + (null_delta >= delta).sum(axis=0)) / (N_PERMUTATIONS + 1)
    q_two, q_right = bh_fdr(p_two), bh_fdr(p_right)
    for j, label in enumerate(block_labels):
        rows.append(dict(mapping=mapping, network_block=label, delta_high_minus_low=delta[j],
            p_two_sided=p_two[j], q_two_sided=q_two[j], p_right=p_right[j], q_right=q_right[j],
            n_permutations=N_PERMUTATIONS))
between_range = pd.DataFrame(rows)
between_range.to_csv(RESULT_DIR / f'classic_notebook_between_range_{N_PERMUTATIONS}perm.csv', index=False)
display(between_range.query("mapping == 'classic_slope'").sort_values('p_two_sided').head(8).round(6))
print('These contrasts test the joint no-association null, not equality of population patterns.')

,mapping,network_block,delta_high_minus_low,p_two_sided,q_two_sided,p_right,q_right,n_permutations
54,classic_slope,DAN-FPN,0.002836,0.041696,0.747198,0.021498,0.393036,10000
63,classic_slope,Lim-FPN,0.002220,0.090191,0.747198,0.045995,0.393036,10000
55,classic_slope,DAN-DMN,0.001906,0.108589,0.747198,0.055294,0.393036,10000
64,classic_slope,Lim-DMN,0.001833,0.118988,0.747198,0.061194,0.393036,10000
60,classic_slope,VAN-DMN,0.001731,0.123888,0.747198,0.062894,0.393036,10000
46,classic_slope,SMN-VAN,0.001552,0.158884,0.747198,0.079692,0.393036,10000
57,classic_slope,VAN-VAN,0.001367,0.197880,0.747198,0.101890,0.393036,10000
62,classic_slope,Lim-Lim,0.001618,0.201880,0.747198,0.100590,0.393036,10000


These contrasts test the joint no-association null, not equality of population patterns.


## 6. 保存经典 Haufe 的模块矩阵与分析说明

In [6]:
networks = ['Vis', 'SMN', 'DAN', 'VAN', 'Lim', 'FPN', 'DMN', 'Subcortex']
for group in GROUPS:
    matrix = pd.DataFrame(np.nan, index=networks, columns=networks)
    for label, value in zip(block_labels, observed_by_group[group][1]):
        a, b = label.split('-')
        matrix.loc[a, b] = matrix.loc[b, a] = value
    matrix.to_csv(RESULT_DIR / f'classic_notebook_{group}_slope_block_matrix.csv')
metadata = dict(
    source_directory=str(SOURCE_DIR), n_permutations=N_PERMUTATIONS,
    permutation_indices=f'0-{N_PERMUTATIONS-1}',
    mapping='selected-space Cov(X, fitted_y) / Var(fitted_y)',
    aggregation='zero-fill nonselected edges; sum / possible edges per block; mean across folds',
    within_group_test='two-sided absolute statistic; sitewise residual IAT permutations',
    ridge_penalty='original fold-specific nested-CV alpha held fixed',
    between_range_null='joint absence of within-range FC-IAT association; not equality of patterns',
    right_tail='exploratory high-minus-low > 0',
    fdr='BH across 36 blocks separately for each group/mapping/tail as applicable',
    engine='Scripts/compare_classic_haufe_permutation.py')
(RESULT_DIR / f'classic_notebook_metadata_{N_PERMUTATIONS}perm.json').write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False))
print('Saved analysis tables and classic slope matrices to:', RESULT_DIR)

Saved analysis tables and classic slope matrices to: /Users/yiling/Documents/Codex/HBN_Cdx/tmpt/CPM/CPM_classic_Haufe
